<a href="https://colab.research.google.com/github/dspraneeth07/CognitiveAttackTopology-CAT/blob/main/Notebooks/04_tensor_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ===============================================================
# NOTEBOOK 04 — TRUST TOPOLOGY TENSOR (TTT)
# CAT Framework
# CPU-SAFE IMPLEMENTATION
# ===============================================================

!pip -q install pandas numpy torch pyarrow scikit-learn seaborn matplotlib tqdm

import os
import json
import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

from google.colab import drive

# ===============================================================
# MOUNT DRIVE
# ===============================================================

drive.mount('/content/drive', force_remount=True)

ROOT = Path("/content/drive/MyDrive/CAT_RESEARCH")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

RUN_DIR = ROOT/"runs"/RUN_ID
DATA_DIR = ROOT/"data"

REPORT_DIR = RUN_DIR/"reports"
TENSOR_DIR = RUN_DIR/"tensor"

for p in [RUN_DIR, REPORT_DIR, TENSOR_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Run:",RUN_ID)

# ===============================================================
# LOAD DATASET
# ===============================================================

DATA_PATH = DATA_DIR/"GCT_phase1_100k.parquet"

df = pd.read_parquet(DATA_PATH)

print("Rows:",len(df))

# ===============================================================
# TRUST SIGNAL VECTOR
# ===============================================================

trust_signals = df[[
"urgency_score",
"fear_trigger_score",
"authority_claim",
"trust_manipulation_score"
]].values

# normalize signals
scaler = MinMaxScaler()

trust_signals = scaler.fit_transform(trust_signals)

# ===============================================================
# INDEX MAPPING
# ===============================================================

user_index = {u:i for i,u in enumerate(df["interaction_id"].unique())}
channel_index = {c:i for i,c in enumerate(df["channel"].unique())}

users = len(user_index)
channels = len(channel_index)
signals = trust_signals.shape[1]

print("Tensor dimensions:",users,channels,signals)

# ===============================================================
# SAFE TENSOR CONSTRUCTION
# ===============================================================

T = torch.zeros(users,channels,signals)

batch = 5000

for start in tqdm(range(0,len(df),batch)):

    chunk = df.iloc[start:start+batch]

    ui = chunk["interaction_id"].map(user_index).values
    cj = chunk["channel"].map(channel_index).values
    sig = scaler.transform(chunk[[
    "urgency_score",
    "fear_trigger_score",
    "authority_claim",
    "trust_manipulation_score"
    ]])

    T[ui,cj,:] = torch.tensor(sig,dtype=torch.float32)

print("Tensor constructed")

# ===============================================================
# TENSOR NORMALIZATION
# ===============================================================

T = (T - T.mean())/(T.std()+1e-6)

# ===============================================================
# PCA ON TRUST SIGNALS
# ===============================================================

pca = PCA(n_components=3)

pca_result = pca.fit_transform(trust_signals)

pca_df = pd.DataFrame(pca_result,columns=["PC1","PC2","PC3"])

pca_df.to_csv(REPORT_DIR/"trust_signal_pca_results.csv",index=False)

# ===============================================================
# TENSOR RANK APPROXIMATION
# ===============================================================

# flatten tensor safely
flat_tensor = T.reshape(users*channels,signals).numpy()

pca_rank = PCA().fit(flat_tensor)

explained = pca_rank.explained_variance_ratio_

rank_df = pd.DataFrame({
"component":range(len(explained)),
"variance":explained
})

rank_df.to_csv(REPORT_DIR/"tensor_rank_analysis.csv",index=False)

# ===============================================================
# TENSOR ENERGY
# ===============================================================

energy = torch.sum(T**2,dim=(1,2))

energy_df = pd.DataFrame({
"user_energy":energy.numpy()
})

energy_df.to_csv(REPORT_DIR/"tensor_energy.csv",index=False)

# ===============================================================
# TENSOR ANOMALY DETECTION
# ===============================================================

threshold = energy.mean()+3*energy.std()

anomaly = energy > threshold

anom_df = pd.DataFrame({
"user_index":range(len(anomaly)),
"anomaly":anomaly.numpy()
})

anom_df.to_csv(REPORT_DIR/"tensor_anomaly_detection_report.csv",index=False)

# ===============================================================
# COGNITIVE DISTORTION ENERGY
# ===============================================================

class TrustTensor:

    r"""
    Trust Topology Tensor

    T_{ijk}

    i = user
    j = channel
    k = trust signal

    Represents interaction geometry.
    """

    def __init__(self,tensor):
        self.tensor = tensor

class CognitiveDistortionEnergy:

    r"""
    CDE = Σ ω_{ijk} T_{ijk}

    Implemented using Einstein Summation.
    """

    def __init__(self,weights):

        self.weights = weights

    def __call__(self,T):

        return torch.einsum("ijk,ijk->",T.tensor,self.weights)

# ===============================================================
# TDI SMOOTHING
# ===============================================================

class KalmanFilter:

    def __init__(self):

        self.x=1
        self.P=1
        self.Q=1e-5
        self.R=1e-2

    def update(self,z):

        K=self.P/(self.P+self.R)

        self.x=self.x+K*(z-self.x)

        self.P=(1-K)*self.P+self.Q

        return self.x

class TrustDistortionIndex:

    def __init__(self):

        self.filter=KalmanFilter()

    def __call__(self,cde,signals):

        var=torch.var(signals)

        smooth=self.filter.update(var.item())

        return cde/(smooth+1e-6)

# ===============================================================
# DEMO CAT COMPUTATION
# ===============================================================

weights = torch.randn(users,channels,signals)

tensor = TrustTensor(T)

cde_engine = CognitiveDistortionEnergy(weights)

cde = cde_engine(tensor)

tdi_engine = TrustDistortionIndex()

tdi = tdi_engine(cde,torch.randn(1000))

print("CDE:",float(cde))
print("TDI:",float(tdi))

# ===============================================================
# SAVE TENSOR COMPONENTS
# ===============================================================

torch.save(T,TENSOR_DIR/"trust_tensor.pt")

# ===============================================================
# FINAL REPORT
# ===============================================================

report = {

"users":users,
"channels":channels,
"signals":signals,
"CDE_demo":float(cde),
"TDI_demo":float(tdi)

}

with open(REPORT_DIR/"tensor_geometry_report.json","w") as f:

    json.dump(report,f,indent=4)

print("Notebook 04 completed")

Mounted at /content/drive
Run: 20260307_045314
Rows: 100000
Tensor dimensions: 35574 3 4


  0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
 25%|██▌       | 5/20 [00:00<00:00, 44.58it/s]/usr/local/lib/python3.12/dist-packages/s

Tensor constructed
CDE: -427.8274230957031
TDI: -417.33990478515625
Notebook 04 completed
